## Create SBASNetworks from groups of Sentinel-1 burst data

This notebook demonstrates how to create SBASNetworks of interferometric pairs from groups of adjacent Sentinel-1 burst data. Refer to the [SBASNetwork Tutorial](SBASNetwork.ipynb) for examples of creating SBASNetworks of networks of full Sentinel-1 scenes or individual Sentinel-1 bursts.

In [ ]:
# This is useful if you are working with a dev install of asf_search and experimenting with changes to the codebase
%load_ext autoreload
%autoreload 2

### Create an asf_search.S1MultiBurstGroup object defining a group of Sentinel-1 bursts and subswaths

Refer to [HyP3's mutli-burst guidelines](https://hyp3-docs.asf.alaska.edu/guides/burst_insar_product_guide/#considerations-for-selecting-input-bursts) when assembling a collection of bursts.

S1MultiBurstGroup validation is performed during initialitzation to prevent the creation of invalid multiburst SBASNetworks that result in failed HyP3 on-demand processing jobs.

In [ ]:
import asf_search as asf

multiburst_group = asf.S1MultiBurstGroup(
    bursts=[
    asf.S1MultiBurst("173_370305", ("IW1", "IW2", "IW3")),
    asf.S1MultiBurst("173_370306", ("IW1", "IW2", "IW3")),
    asf.S1MultiBurst("173_370307", ("IW1", "IW2", "IW3"))
    ]
)
multiburst_group

### Create a geoographic reference S1MultiBurstProduct object

An `S1MultiBurstProduct` contains a collection of Sentinel-1 burst products.

In [ ]:
start_date = '2023-01-01'

reference_multiburst = asf.S1MultiBurstProduct(multiburst_group, start_date)
reference_multiburst

### Create an SBASNetwork from the S1MultiBurstProduct object

In [ ]:
from datetime import datetime, date
import pandas as pd

def get_julian_season(season) -> tuple[int,int]:
    season_start_ts = pd.Timestamp(
        datetime.strptime(f"{season[0]}-0001", "%m-%d-%Y"), tz="UTC"
        )
    season_start_day = season_start_ts.timetuple().tm_yday
    season_end_ts = pd.Timestamp(
        datetime.strptime(f"{season[1]}-0001", "%m-%d-%Y"), tz="UTC"
    )
    season_end_day = season_end_ts.timetuple().tm_yday
    return (season_start_day, season_end_day)

season = ("1-1", "6-25")

multiburst_sbas = asf.SBASNetwork.from_geo_reference(
    geo_reference = reference_multiburst,
    start_date = '2023-01-01',
    end_date = '2025-10-02',
    season = get_julian_season(season),
    perpendicular_baseline=150, 
    inseason_temporal_baseline=36,
    bridge_target_date='3-1',
    bridge_year_threshold=1,
    allow_missing_state_vectors=True)

multiburst_sbas.plot()

## Add and remove Pairs from an SBASNetwork 

### Add Pairs to the `SBASNetwork` to manually connect an unused S1MultiBurst

You can add or remove pairs by passing:
- A Tuple of date pair strings: ("2024-02-03", "2024-04-15")
- A List of tuples of date pair strings: [("2024-02-03", "2024-04-15"), ("2024-02-03", "2024-04-15")]
- A Pair object
- A List of Pair objects

In [ ]:
multiburst_sbas.add_pairs([
    ("2024-03-22", "2024-04-15"),
    ("2024-04-03", "2024-04-15")
    ])

multiburst_sbas.plot()

### Add a Pair by passing a Pair object

### 

In [ ]:
pair = multiburst_sbas.remove_list[60]
print(f"{pair.ref_time.date()}, {pair.sec_time.date()}")

multiburst_sbas.add_pairs(pair)

multiburst_sbas.plot()

### Remove Pairs

In [ ]:
multiburst_sbas.remove_pairs([
    ("2024-04-27", "2024-05-21"),
    ("2024-05-21", "2024-06-14"),
    ("2024-06-02", "2024-06-14"),
    ("2024-05-09", "2024-06-14"),
    ("2024-05-09", "2024-06-02"),
    ("2024-04-27", "2024-06-02"),
    ("2024-04-27", "2024-05-09")
])

multiburst_sbas.plot()

<hr>

## Use the `scene_ids` property to access multi-burst product IDs for easy InSAR product ordering from HyP3 

The `scene_ids` property provides scene IDs for the largest network in the `connected_substacks` list for each SBASNetwork in `sbas_networks`

An SBAS Network generated from a `S1MultiBurstProduct` geo-reference can be used to order Sentinel-1 multi-burst InSAR processing from ASF [HyP3](https://hyp3-docs.asf.alaska.edu/hyp3-docs/guides/burst_insar_product_guide/) and [HyP3+](https://hyp3-docs.asf.alaska.edu/hyp3-docs/about/hyp3_plus/)

In [ ]:
sbas_burst_ids = multiburst_sbas.scene_ids
sbas_burst_ids

<hr>

## Use the `get_scene_ids` method to access multi-burst product IDs from any of the SBASNetwork pair lists in `sbas_networks`

Options include:
- `SBASNetwork.full_stack`
- `SBASNetwork.remove_list`
- The possibly disconnected `SBASNetwork.subset_stack`
- Any of the stack_lists in `SBASNetwork.connected_substacks`



In [ ]:
full_stack_burst_ids = multiburst_sbas.get_scene_ids(multiburst_sbas.full_stack)
full_stack_burst_ids